# Investigate and Stress-Test a Fraud Detection System

A fraud model has been trained, but a good score on one dataset is not enough. The fraud team wants to know whether the model survives changing timelines, coordinated networks, subtle behavior, and realistic what-if cases. FraudTwin gives us controlled experiments where we know exactly what changed.

## 1. Set up the investigation

We begin with versioned benchmark configurations rather than hand-made examples. FraudTwin records the seed, configuration, and output fingerprints in each manifest, so another engineer can repeat the same investigation later.

In [1]:
from datetime import timedelta
from pathlib import Path
from tempfile import TemporaryDirectory

import polars as pl

import fraudtwin
from fraudtwin.config import load_config
from fraudtwin.graph import build_graph
from fraudtwin.replay import replay_run

project_root = Path.cwd().parent.parent
config_dir = project_root / "configs" / "benchmarks"
graph_config = load_config(config_dir / "m13-camouflage-v1.yaml")
difficulty_config = load_config(config_dir / "m12-difficulty-v1.yaml")
counterfactual_config = load_config(config_dir / "m14-counterfactual-v1.yaml")

## 2. Establish a fraud baseline

First we create the reference world: ordinary payments plus known fraud campaigns. This is our control group; without it, we could not tell whether a later stress test changed the model's challenge or simply changed the whole dataset.

In [2]:
baseline = fraudtwin.generate(graph_config)

print("Run:", baseline.run_id)
print("Payments:", len(baseline.behavior.payments))
print("Fraud records:", len(baseline.behavior.fraud_records))

Run: RUN-76d5222df8f299f6
Payments: 949
Fraud records: 108


## 3. Replay the incident

Imagine an investigator asking, 'What did the system see during the first six hours?' Replay answers that question from the recorded run, preserving event identity and ordering instead of generating a new story. The temporary directory is only the notebook's local run archive.

In [3]:
with TemporaryDirectory(prefix="fraudtwin-investigation-") as output_dir:
    written = fraudtwin.generate(graph_config, write=True, output_dir=output_dir)
    replay = replay_run(
        written.run_dir,
        graph_config.simulation.start,
        graph_config.simulation.start + timedelta(hours=6),
    )

    print("Replayed events:", replay.count)
    replay.frame.select(["replay_sequence", "event_type", "payment_id", "event_time"]).head()

Replayed events: 331


## 4. Measure the fraud graph

Some fraud is not suspicious in one payment but becomes suspicious as a relationship pattern. FraudTwin lets us compare the observable graph available to a detector with the oracle graph that contains the simulator's complete campaign truth.

In [4]:
observable = build_graph(
    graph_config, baseline.entities, baseline.behavior, baseline.manifest, view="observable"
)
oracle = build_graph(
    graph_config, baseline.entities, baseline.behavior, baseline.manifest, view="oracle"
)

pl.DataFrame(
    {
        "View": ["Observable", "Oracle"],
        "Nodes": [len(observable.nodes), len(oracle.nodes)],
        "Edges": [len(observable.edges), len(oracle.edges)],
        "Patterns": [len(observable.patterns), len(oracle.patterns)],
    }
)

View,Nodes,Edges,Patterns
str,i64,i64,i64
"""Observable""",417,5081,1072
"""Oracle""",443,5216,1095


## 5. Make fraud harder to recognize

If the baseline model succeeds only because fraud is extreme, it is not ready for production. The difficulty profile makes fraud more similar to legitimate behavior while keeping the underlying fraud objective intact.

In [5]:
harder = fraudtwin.generate(difficulty_config)

difficulty = harder.manifest.difficulty
pl.DataFrame(
    {
        "Run": [baseline.run_id, harder.run_id],
        "Fraud records": [len(baseline.behavior.fraud_records), len(harder.behavior.fraud_records)],
        "Difficulty": [
            "camouflage benchmark",
            str(difficulty["requested_difficulty"]) if difficulty else "none",
        ],
    }
)

Run,Fraud records,Difficulty
str,i64,str
"""RUN-76d5222df8f299f6""",108,"""camouflage benchmark"""
"""RUN-642709d737867d7f""",34,"""7"""


## 6. Add feature and relation camouflage

Difficulty changes how subtle the fraud is; camouflage changes what it resembles. Here, FraudTwin can make amounts, timing, devices, and relationships look ordinary, while the manifest records the exact strength used.

In [6]:
camouflage = baseline.manifest.camouflage
pl.DataFrame(
    {
        "Setting": ["Global", "Feature", "Relation"],
        "Strength": [
            camouflage["resolved_global"]["camouflage"],
            camouflage["resolved_global"]["feature"],
            camouflage["resolved_global"]["relation"],
        ],
    }
)

Setting,Strength
str,f64
"""Global""",0.8
"""Feature""",0.85
"""Relation""",0.75


## 7. Ask a counterfactual question

Finally, the team asks a practical question: what small change could turn a legitimate payment trajectory into a selected fraud pattern? Counterfactuals answer that question and keep links to the original payment, events, and ledger records.

In [7]:
counterfactual_run = fraudtwin.generate(counterfactual_config)
counterfactual = counterfactual_run.behavior.counterfactual

pl.DataFrame(
    [
        {
            "Objective": change.objective,
            "Status": change.status,
            "Source payment": change.source_payment_id,
            "Derived payment": change.derived_payment_id,
            "Distance": change.effective_distance,
            "Changed dimensions": len(change.changed_fields),
        }
        for change in counterfactual.change_sets
    ]
)

Objective,Status,Source payment,Derived payment,Distance,Changed dimensions
str,str,str,str,f64,i64
"""F01""","""ACCEPTED""","""PAY-00000670""","""CF-PAY-ae3db91708dcf93fd927""",1.0,1
"""F03""","""ACCEPTED""","""PAY-00000434""","""CF-PAY-0267e083b9c5f414158e""",1.0,1
"""F04""","""ACCEPTED""","""PAY-00000155""","""CF-PAY-7df8b9e1c7893631bd62""",1.0,1


## 8. Compare the evidence

The investigation ends with evidence, not one magic metric. Each run has its own identity, but every result remains connected to payments, events, graph relationships, and fraud objectives, making failures easier to explain.

In [8]:
pl.DataFrame(
    {
        "Experiment": ["Baseline", "Harder fraud", "Counterfactual"],
        "Run ID": [baseline.run_id, harder.run_id, counterfactual_run.run_id],
        "Fraud records": [
            len(baseline.behavior.fraud_records),
            len(harder.behavior.fraud_records),
            len(counterfactual.fraud_records),
        ],
    }
)

Experiment,Run ID,Fraud records
str,str,i64
"""Baseline""","""RUN-76d5222df8f299f6""",108
"""Harder fraud""","""RUN-642709d737867d7f""",34
"""Counterfactual""","""RUN-5ee94cd664213302""",3


## References and implementation notes

- [Fraud Detection Handbook - simulated datasets](https://fraud-detection-handbook.github.io/fraud-detection-handbook/Chapter_3_GettingStarted/SimulatedDataset.html) - informed the scenario-based fraud generation and the use of legitimate behavior as the baseline.
- [Fraud Detection Handbook - validation strategies](https://fraud-detection-handbook.github.io/fraud-detection-handbook/Chapter_5_ModelValidationAndSelection/ValidationStrategies.html) - informed temporal replay, delayed-label gaps, and future-oriented evaluation.
- [Feast - point-in-time joins](https://docs.feast.dev/getting-started/concepts/point-in-time-joins) - informed the separation of event time, feature availability, and label availability used by FraudTwin's ML datasets.
- [IBM AMLSim](https://github.com/IBM/AMLSim) - provided a reference for synthetic banking networks and known fraud relationships.
- [Santander Gen-Fraud-Graph](https://github.com/SantanderAI/gen-fraud-graph) - informed reproducible graph campaigns, fraud-ring structures, and graph-oriented evaluation.
- [Mothilal, Sharma, and Tan - counterfactual explanations](https://doi.org/10.1145/3351095.3372850) - informed minimum-change counterfactuals, feasibility constraints, and change explanations.

These sources were used as design references, not copied datasets or implementations. FraudTwin combines their relevant ideas into one deterministic workflow: replay the same run, compare observable and oracle evidence, increase fraud difficulty, and explain counterfactual changes.